# LeetCode 23 — Merge k Sorted Lists bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:

```

Mục tiêu của bài học:

1. Hiểu **linked list** (danh sách liên kết) từ con số 0: node, `val`, `next`, `None`, cách duyệt, kỹ thuật **dummy head**.
2. Viết được "viên gạch" quan trọng nhất: **merge two sorted lists** (gộp 2 list đã sắp xếp).
3. Đi qua 5 cách giải, từ trực giác đến tối ưu, và hiểu **cách cũ vướng ở đâu → cải tiến ra sao**.
4. Hiểu **heap / `heapq`** và mẹo tuple `(val, idx, node)` để không phải so sánh hai `ListNode`.
5. Hiểu **chia để trị** (divide and conquer) ghép cặp, đạt `O(N log k)`.
6. Tự viết lại lời giải, tránh các lỗi kinh điển và nhận dạng pattern "gộp k nguồn đã sắp xếp".

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một mảng `lists` gồm `k` linked list, mỗi linked list đã được **sắp xếp tăng dần**.
>
> Hãy gộp (merge) tất cả các linked list thành **một** linked list đã sắp xếp và trả về nó.

**Example 1**

```text
Input: lists = [[1,4,5],[1,3,4],[2,6]]
Output: [1,1,2,3,4,4,5,6]
Giải thích: các linked list là
[
  1->4->5,
  1->3->4,
  2->6
]
gộp lại thành một linked list đã sắp xếp:
1->1->2->3->4->4->5->6
```

**Example 2**

```text
Input: lists = []
Output: []
```

**Example 3**

```text
Input: lists = [[]]
Output: []
```

**Constraints**

- `k == lists.length`
- `0 <= k <= 10^4`
- `0 <= lists[i].length <= 500`
- `-10^4 <= lists[i][j] <= 10^4`
- `lists[i]` được sắp xếp **tăng dần**.
- Tổng của các `lists[i].length` không vượt quá `10^4`.

### Ký hiệu dùng suốt notebook

- `k` = số linked list trong `lists` (có thể bằng 0).
- `N` = **tổng số node** của tất cả các list cộng lại (theo đề, `N <= 10^4`).

Quan sát nhanh trước khi code:

| Input | Điều cần để ý |
|---|---|
| `[[1,4,5],[1,3,4],[2,6]]` | có giá trị **trùng nhau** (hai số `1`, hai số `4`) |
| `[]` | `k = 0`, không có list nào → trả `None` |
| `[[]]` | `k = 1` nhưng list đó **rỗng** → phần tử là `None` → trả `None` |

Chú ý: trên LeetCode, `[[1,4,5], ...]` chỉ là cách *in* cho dễ đọc. Thật ra mỗi phần tử của `lists` là **node đầu** (head) của một linked list, hoặc `None` nếu list rỗng. Vậy linked list là gì? Ta học ngay sau đây.

## 2. Nền tảng: Linked list từ căn bản

### 2.1 Node là gì?

Python `list` như `[1, 4, 5]` lưu các phần tử **liền nhau** và truy cập bằng chỉ số `a[0]`, `a[1]`...

**Linked list** thì khác: mỗi phần tử là một **node** (nút) riêng, gồm 2 thứ:

- `val`: giá trị của node.
- `next`: "mũi tên" trỏ tới node kế tiếp, hoặc `None` nếu là node cuối.

```text
 head
  |
  v
[1 | next] ---> [4 | next] ---> [5 | None]
```

Ta chỉ cần giữ **node đầu** (`head`) là đi được tới mọi node bằng cách đi theo `next`.

LeetCode định nghĩa node bằng class `ListNode` dưới đây. Ta tự định nghĩa lại **y hệt** để chạy trong notebook.

In [1]:
from typing import List, Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


# Tạo tay linked list 1 -> 4 -> 5
n5 = ListNode(5)          # node cuối, next mặc định là None
n4 = ListNode(4, n5)      # 4 trỏ tới 5
n1 = ListNode(1, n4)      # 1 trỏ tới 4
head = n1

print(head.val)            # 1
print(head.next.val)       # 4
print(head.next.next.val)  # 5
print(head.next.next.next) # None -> hết list

1
4
5
None


Nhận xét:

- `ListNode(5)` dùng giá trị mặc định `next=None`.
- `head.next.next.val` đọc là: "từ head, đi theo next 2 lần, lấy val".
- Gặp `None` nghĩa là **đã hết list**.

Dòng `from typing import List, Optional` dùng cho **type hint** (gợi ý kiểu) trong signature:

- `List[X]`: một list chứa các phần tử kiểu `X`.
- `Optional[ListNode]`: hoặc là một `ListNode`, hoặc là `None`.

Vậy `lists: List[Optional[ListNode]]` nghĩa là "một list, mỗi phần tử là head của một linked list hoặc `None`". Trên LeetCode các tên này đã được import sẵn; trong notebook ta phải tự import.

### 2.2 Duyệt linked list bằng `while cur`

Không có chỉ số `i`, nên ta dùng một con trỏ `cur` (current) đi dần:

```python
cur = head
while cur:          # còn node thì còn chạy
    print(cur.val)
    cur = cur.next  # bước sang node kế tiếp
```

Vì sao `while cur:` chạy được? Trong Python, `None` được coi là **False**, còn một object `ListNode` được coi là **True**. Nên `while cur:` nghĩa là "khi `cur` chưa phải `None`".

In [2]:
print(bool(None))          # False
print(bool(ListNode(0)))   # True (kể cả val = 0, vì ta hỏi về node, không phải val)

cur = head
step = 0
while cur:
    print(f"bước {step}: cur.val = {cur.val}")
    cur = cur.next
    step += 1
print("cur bây giờ là:", cur)

False
True
bước 0: cur.val = 1
bước 1: cur.val = 4
bước 2: cur.val = 5
cur bây giờ là: None


### 2.3 Helper `build_list`, `to_list`, `list_str` để test

Tạo tay từng node rất mệt. Ta viết 3 hàm phụ (helper):

- `build_list([1, 4, 5])` → trả về head của `1 -> 4 -> 5` (list rỗng → `None`).
- `to_list(head)` → đổi ngược linked list thành Python list `[1, 4, 5]` để so sánh bằng `assert`.
- `list_str(head)` → chuỗi `"1->4->5"` để in cho đẹp (`None` → `"∅"`).
- `build_lists([[1,4,5],[2,6]])` → list các head, đúng dạng input `lists` của LeetCode.

`build_list` dùng kỹ thuật **dummy head** — sẽ giải thích ngay ở mục 2.4.

In [3]:
def build_list(values):
    dummy = ListNode()          # node giả đứng trước node thật đầu tiên
    tail = dummy                # tail luôn là node cuối của list đang dựng
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next           # bỏ node giả, trả node thật đầu tiên


def to_list(head):
    out = []
    cur = head
    while cur:
        out.append(cur.val)
        cur = cur.next
    return out


def list_str(head):
    return "->".join(str(v) for v in to_list(head)) if head else "∅"


def build_lists(list_of_values):
    return [build_list(values) for values in list_of_values]


h = build_list([1, 4, 5])
print(list_str(h), to_list(h))
print(build_list([]), list_str(build_list([])))
print([list_str(x) for x in build_lists([[1, 4, 5], [1, 3, 4], [2, 6], []])])

1->4->5 [1, 4, 5]
None ∅
['1->4->5', '1->3->4', '2->6', '∅']


### 2.4 Kỹ thuật dummy head — vì sao cần?

Khi dựng một list mới bằng cách nối từng node vào **cuối**, ta giữ con trỏ `tail` (đuôi).

Không có dummy, node đầu tiên là trường hợp đặc biệt: lúc đầu chưa có `head`, chưa có `tail`, nên phải `if head is None: ...`.

Với dummy (node giả, giá trị gì cũng được):

```text
dummy -> (chưa có gì)
  ^
 tail

nối 1:   dummy -> 1           tail ở 1
nối 4:   dummy -> 1 -> 4      tail ở 4
kết quả: dummy.next = 1 -> 4
```

Mọi node đều được nối **cùng một cách** `tail.next = node; tail = tail.next`, không còn trường hợp đặc biệt. Cuối cùng trả `dummy.next`.

So sánh hai cách viết:

In [4]:
def build_without_dummy(values):
    head = None
    tail = None
    for v in values:
        node = ListNode(v)
        if head is None:        # trường hợp đặc biệt cho node đầu
            head = node
            tail = node
        else:
            tail.next = node
            tail = node
    return head


def build_with_dummy(values):
    dummy = ListNode(-999)      # giá trị dummy không quan trọng
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next           # KHÔNG trả dummy


print(list_str(build_without_dummy([1, 4, 5])))
print(list_str(build_with_dummy([1, 4, 5])))
print(list_str(build_with_dummy([])))   # list rỗng -> dummy.next là None

1->4->5
1->4->5
∅


## 3. Viên gạch nền: Merge Two Sorted Lists (LeetCode 21)

Hầu hết các cách giải bài 23 đều dựa trên việc **gộp 2 list đã sắp xếp**. Học kỹ phần này trước.

Ý tưởng: hai con trỏ `a`, `b` đứng ở đầu hai list.

- So sánh `a.val` và `b.val`, node nào **nhỏ hơn** thì gắn vào đuôi kết quả và con trỏ đó tiến lên.
- Khi một list đã hết (`None`), phần còn lại của list kia **đã sắp xếp sẵn** → gắn nguyên cụm vào đuôi.

Điểm hay: ta **không tạo node mới**, chỉ nối lại các mũi tên `next` của node có sẵn.

Hai cú pháp mới:

- `while a and b:` chạy khi **cả hai** chưa phải `None`.
- `tail.next = a if a else b`: nếu `a` còn node thì gắn `a`, ngược lại gắn `b` (có thể là `None`).

In [5]:
def merge_two_lists(a, b):
    dummy = ListNode()
    tail = dummy
    while a and b:
        if a.val <= b.val:      # <= : khi bằng nhau, ưu tiên list a
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b   # gắn phần còn lại (đã sắp xếp sẵn)
    return dummy.next


print(list_str(merge_two_lists(build_list([1, 4, 5]), build_list([1, 3, 4]))))
print(list_str(merge_two_lists(build_list([2, 6]), None)))
print(list_str(merge_two_lists(None, None)))

1->1->3->4->4->5
2->6
∅


### Trace gộp `1->4->5` và `1->3->4`

| Bước | `a.val` | `b.val` | So sánh | Lấy node | Kết quả sau bước |
|---:|---:|---:|---|---|---|
| 1 | 1 | 1 | `1 <= 1` đúng | `a` (1) | `1` |
| 2 | 4 | 1 | `4 <= 1` sai | `b` (1) | `1->1` |
| 3 | 4 | 3 | `4 <= 3` sai | `b` (3) | `1->1->3` |
| 4 | 4 | 4 | `4 <= 4` đúng | `a` (4) | `1->1->3->4` |
| 5 | 5 | 4 | `5 <= 4` sai | `b` (4) | `1->1->3->4->4` |
| hết | 5 | None | `b` hết | gắn phần còn lại của `a` | `1->1->3->4->4->5` |

Hàm `trace_merge_two` in đúng các trạng thái này:

In [6]:
def trace_merge_two(a, b):
    dummy = ListNode()
    tail = dummy
    step = 1
    while a and b:
        if a.val <= b.val:
            took = f"a ({a.val})"
            tail.next = a
            a = a.next
        else:
            took = f"b ({b.val})"
            tail.next = b
            b = b.next
        tail = tail.next
        # cắt tạm để in phần kết quả đã có (không làm hỏng vì tail.next sẽ được gán lại)
        saved, tail.next = tail.next, None
        print(f"Bước {step}: lấy {took:6} | kết quả = {list_str(dummy.next)}")
        tail.next = saved
        step += 1
    rest = a if a else b
    print(f"Một list đã hết -> gắn phần còn lại: {list_str(rest)}")
    tail.next = rest
    print("Kết quả cuối:", list_str(dummy.next))


trace_merge_two(build_list([1, 4, 5]), build_list([1, 3, 4]))

Bước 1: lấy a (1)  | kết quả = 1
Bước 2: lấy b (1)  | kết quả = 1->1
Bước 3: lấy b (3)  | kết quả = 1->1->3
Bước 4: lấy a (4)  | kết quả = 1->1->3->4
Bước 5: lấy b (4)  | kết quả = 1->1->3->4->4
Một list đã hết -> gắn phần còn lại: 5
Kết quả cuối: 1->1->3->4->4->5


**Độ phức tạp của merge two**: nếu hai list dài `n1` và `n2` thì mỗi vòng lặp lấy đúng 1 node → tối đa `n1 + n2` vòng.

- Time: `O(n1 + n2)`.
- Space: `O(1)` bộ nhớ phụ (chỉ vài con trỏ, dùng lại node cũ).

Lưu ý quan trọng: hàm này **sửa `next` của các node đầu vào**. Sau khi merge, list `a` cũ không còn nguyên vẹn. Khi test nhiều cách giải, nhớ **dựng lại input mới** cho mỗi lần gọi.

## 4. Cách 1 — Gom hết giá trị → sort → dựng lại

Cách người mới nghĩ ra đầu tiên: "quên linked list đi, cứ lấy hết số ra rồi sort".

1. Duyệt từng list, `append` mọi `val` vào một Python list `values`.
2. `values.sort()`.
3. Dựng linked list mới từ `values` (dùng dummy head).

Ví dụ 1: `values = [1,4,5,1,3,4,2,6]` → sort → `[1,1,2,3,4,4,5,6]` → dựng lại.

In [7]:
def merge_k_sort_values(lists):
    values = []
    for head in lists:
        cur = head
        while cur:
            values.append(cur.val)
            cur = cur.next
    values.sort()
    return build_list(values)


print(list_str(merge_k_sort_values(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))
print(list_str(merge_k_sort_values(build_lists([]))))
print(list_str(merge_k_sort_values(build_lists([[]]))))

1->1->2->3->4->4->5->6
∅
∅


### Độ phức tạp cách 1 và chỗ vướng

- Time: `O(N log N)` — gom `O(N)`, sort `O(N log N)`, dựng lại `O(N)`.
- Space: `O(N)` — mảng `values` và `N` node mới.

Với `N <= 10^4`, cách này thực tế vẫn được LeetCode chấp nhận. Nhưng nó **lãng phí một dữ kiện quan trọng**: từng list *đã được sắp xếp sẵn*. Sort lại từ đầu như thể dữ liệu hỗn loạn là bỏ phí thông tin, và còn tốn thêm `O(N)` bộ nhớ.

**Câu hỏi dẫn đường**: ta đã biết gộp 2 list sorted trong `O(n1 + n2)`. Có tận dụng được để gộp `k` list không?

## 5. Cách 2 — Merge lần lượt từng list vào kết quả

Ý tưởng: giống cộng dồn một tổng.

```python
result = None
for head in lists:
    result = merge_two_lists(result, head)
```

Ví dụ 1:

| Lần | Gộp thêm | `result` sau khi gộp | Độ dài `result` |
|---:|---|---|---:|
| 1 | `1->4->5` | `1->4->5` | 3 |
| 2 | `1->3->4` | `1->1->3->4->4->5` | 6 |
| 3 | `2->6` | `1->1->2->3->4->4->5->6` | 8 |

Trong hàm trace bên dưới có cú pháp mới `enumerate`: duyệt một list mà lấy được cả **chỉ số** lẫn **giá trị**. Mặc định chỉ số bắt đầu từ `0`; viết `enumerate(x, start=1)` thì bắt đầu từ `1`. Thử nhanh:

In [8]:
for i, word in enumerate(["a", "b", "c"]):
    print(i, word)
print(list(enumerate(["a", "b"], start=1)))

0 a
1 b
2 c
[(1, 'a'), (2, 'b')]


In [9]:
def merge_k_one_by_one(lists):
    result = None
    for head in lists:
        result = merge_two_lists(result, head)
    return result


def trace_one_by_one(lists):
    result = None
    for i, head in enumerate(lists, start=1):
        incoming = list_str(head)
        result = merge_two_lists(result, head)
        print(f"Lần {i}: gộp thêm {incoming:8} -> result = {list_str(result)} (dài {len(to_list(result))})")
    return result


trace_one_by_one(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))
print(list_str(merge_k_one_by_one(build_lists([]))), list_str(merge_k_one_by_one(build_lists([[]]))))

Lần 1: gộp thêm 1->4->5  -> result = 1->4->5 (dài 3)
Lần 2: gộp thêm 1->3->4  -> result = 1->1->3->4->4->5 (dài 6)
Lần 3: gộp thêm 2->6     -> result = 1->1->2->3->4->4->5->6 (dài 8)
∅ ∅


### Độ phức tạp cách 2 và chỗ vướng

Mỗi lần gộp phải đi lại **toàn bộ `result`** — mà `result` mỗi lúc một dài.

Giả sử có `k` list, mỗi list `n` node (`N = k·n`):

- Lần 1 tốn khoảng `n`, lần 2 khoảng `2n`, ..., lần `k` khoảng `k·n`.
- Tổng ≈ `n·(1 + 2 + ... + k) = n·k(k+1)/2` ≈ `k·N / 2`.

Vậy:

- Time: `O(k·N)`.
- Space: `O(1)` bộ nhớ phụ.

Trường hợp xấu theo constraints: `k = 10^4` list, mỗi list 1 node → khoảng `5·10^7` bước, Python dễ bị **TLE** (Time Limit Exceeded).

**Chỗ vướng**: những node nhỏ đầu tiên bị duyệt đi duyệt lại ở *mọi* lần gộp. Ta sẽ sửa điều này bằng hai hướng: (a) chọn node nhỏ nhất trực tiếp trong `k` đầu list (cách 3 → cách 4), (b) gộp theo cặp cho cân bằng (cách 5).

## 6. Cách 3 — So sánh `k` đầu list ở mỗi bước

Mở rộng tự nhiên của merge two: thay vì 2 con trỏ, ta có **k con trỏ** `heads[0..k-1]`, mỗi con trỏ ở đầu phần còn lại của một list.

Mỗi bước:

1. Quét qua `k` con trỏ, tìm node có `val` **nhỏ nhất** (bỏ qua con trỏ đã `None`).
2. Gắn node đó vào đuôi kết quả, con trỏ đó tiến lên `next`.
3. Nếu mọi con trỏ đều `None` → xong.

Ta dùng `enumerate` (đã học ở mục 5) để biết node đang xét thuộc list số mấy.

In [10]:
def merge_k_compare_heads(lists):
    heads = list(lists)              # copy mảng con trỏ (không copy node)
    dummy = ListNode()
    tail = dummy
    while True:
        best = -1                    # chỉ số list đang có đầu nhỏ nhất
        for i, node in enumerate(heads):
            if node and (best == -1 or node.val < heads[best].val):
                best = i
        if best == -1:               # mọi list đều đã hết
            break
        tail.next = heads[best]
        tail = tail.next
        heads[best] = heads[best].next
    return dummy.next


print(list_str(merge_k_compare_heads(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))
print(list_str(merge_k_compare_heads(build_lists([]))), list_str(merge_k_compare_heads(build_lists([[]]))))

1->1->2->3->4->4->5->6
∅ ∅


### Trace cách 3 với Example 1

Cột "đầu các list" là `heads[0], heads[1], heads[2]` trước khi chọn (`None` = list đã hết). Dùng `<` nên khi bằng nhau, list có chỉ số nhỏ hơn thắng.

| Bước | Đầu các list | Chọn (list, val) | Kết quả |
|---:|---|---|---|
| 1 | `[1, 1, 2]` | list 0, 1 | `1` |
| 2 | `[4, 1, 2]` | list 1, 1 | `1->1` |
| 3 | `[4, 3, 2]` | list 2, 2 | `1->1->2` |
| 4 | `[4, 3, 6]` | list 1, 3 | `1->1->2->3` |
| 5 | `[4, 4, 6]` | list 0, 4 | `...->4` |
| 6 | `[5, 4, 6]` | list 1, 4 | `...->4->4` |
| 7 | `[5, None, 6]` | list 0, 5 | `...->5` |
| 8 | `[None, None, 6]` | list 2, 6 | `1->1->2->3->4->4->5->6` |

In [11]:
def trace_compare_heads(lists):
    heads = list(lists)
    out = []
    step = 1
    while True:
        snapshot = [h.val if h else None for h in heads]
        best = -1
        for i, node in enumerate(heads):
            if node and (best == -1 or node.val < heads[best].val):
                best = i
        if best == -1:
            print(f"Đầu các list = {snapshot} -> tất cả đã hết, dừng.")
            break
        out.append(heads[best].val)
        print(f"Bước {step}: đầu = {snapshot} -> chọn list {best} (val {heads[best].val}) | kết quả = {out}")
        heads[best] = heads[best].next
        step += 1


trace_compare_heads(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))

Bước 1: đầu = [1, 1, 2] -> chọn list 0 (val 1) | kết quả = [1]
Bước 2: đầu = [4, 1, 2] -> chọn list 1 (val 1) | kết quả = [1, 1]
Bước 3: đầu = [4, 3, 2] -> chọn list 2 (val 2) | kết quả = [1, 1, 2]
Bước 4: đầu = [4, 3, 6] -> chọn list 1 (val 3) | kết quả = [1, 1, 2, 3]
Bước 5: đầu = [4, 4, 6] -> chọn list 0 (val 4) | kết quả = [1, 1, 2, 3, 4]
Bước 6: đầu = [5, 4, 6] -> chọn list 1 (val 4) | kết quả = [1, 1, 2, 3, 4, 4]
Bước 7: đầu = [5, None, 6] -> chọn list 0 (val 5) | kết quả = [1, 1, 2, 3, 4, 4, 5]
Bước 8: đầu = [None, None, 6] -> chọn list 2 (val 6) | kết quả = [1, 1, 2, 3, 4, 4, 5, 6]
Đầu các list = [None, None, None] -> tất cả đã hết, dừng.


### Độ phức tạp cách 3 và chỗ vướng

- Có `N` bước (mỗi bước lấy đúng 1 node), mỗi bước quét `k` con trỏ → Time: `O(k·N)`.
- Space: `O(k)` cho mảng `heads` (hoặc `O(1)` nếu sửa thẳng trên `lists`).

So với cách 2: không còn duyệt lại kết quả, nhưng **việc tìm min trong k đầu list** vẫn tốn `O(k)` mỗi bước.

**Mấu chốt cải tiến**: giữa hai bước liên tiếp, tập "k đầu list" chỉ thay đổi **một phần tử** (lấy ra một node, thêm vào node kế tiếp của nó). Quét lại cả `k` phần tử là phí. Ta cần một cấu trúc dữ liệu:

- lấy ra phần tử nhỏ nhất thật nhanh,
- thêm một phần tử mới thật nhanh.

Đó chính là **heap** (hàng đợi ưu tiên — priority queue).

## 7. Nền tảng: Heap và module `heapq`

### 7.1 Heap là gì?

**Min-heap** là một cây nhị phân (mỗi node có tối đa 2 con) thỏa: **cha luôn ≤ các con**. Hệ quả: phần tử **nhỏ nhất luôn ở gốc**.

Python lưu heap trong một list thường, theo quy ước vị trí:

- gốc ở `heap[0]`,
- con của `heap[i]` là `heap[2*i + 1]` và `heap[2*i + 2]`.

Module `heapq` cung cấp:

| Hàm | Làm gì | Time (heap có `m` phần tử) |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` vào heap | `O(log m)` |
| `heapq.heappop(h)` | lấy ra và xóa phần tử **nhỏ nhất** | `O(log m)` |
| `h[0]` | xem phần tử nhỏ nhất (không xóa) | `O(1)` |
| `heapq.heapify(h)` | biến list bất kỳ thành heap tại chỗ | `O(m)` |

Lưu ý: heap **không** phải list đã sort hoàn toàn; nó chỉ đảm bảo gốc nhỏ nhất.

In [12]:
import heapq

h = []
for x in [5, 1, 8, 3, 2]:
    heapq.heappush(h, x)
    print(f"push {x} -> heap (dạng list) = {h}, nhỏ nhất h[0] = {h[0]}")

print("Pop lần lượt:", [heapq.heappop(h) for _ in range(5)])

arr = [9, 4, 7, 1]
heapq.heapify(arr)
print("heapify [9, 4, 7, 1] ->", arr)

push 5 -> heap (dạng list) = [5], nhỏ nhất h[0] = 5
push 1 -> heap (dạng list) = [1, 5], nhỏ nhất h[0] = 1
push 8 -> heap (dạng list) = [1, 5, 8], nhỏ nhất h[0] = 1
push 3 -> heap (dạng list) = [1, 3, 8, 5], nhỏ nhất h[0] = 1
push 2 -> heap (dạng list) = [1, 2, 8, 5, 3], nhỏ nhất h[0] = 1
Pop lần lượt: [1, 2, 3, 5, 8]
heapify [9, 4, 7, 1] -> [1, 4, 7, 9]


Quan sát:

- Sau `push 3`, list là `[1, 3, 8, 5]` — không sort hoàn toàn, nhưng `h[0] = 1` luôn nhỏ nhất.
- Pop lần lượt ra `[1, 2, 3, 5, 8]` — tăng dần. Heap = "máy lấy min".

### 7.2 So sánh tuple — heap chứa được gì?

Heap cần so sánh các phần tử bằng `<`. Python so sánh **tuple** theo kiểu từ điển (lexicographic): so phần tử đầu, nếu bằng nhau thì so phần tử thứ hai, rồi thứ ba...

In [13]:
print((1, 5) < (2, 0))     # True : 1 < 2, không cần xem tiếp
print((1, 5) < (1, 7))     # True : 1 == 1, so tiếp 5 < 7
print((4, 0) < (4, 0))     # False: bằng nhau hoàn toàn

h = []
for item in [(4, "list0"), (1, "list1"), (2, "list2")]:
    heapq.heappush(h, item)
print(heapq.heappop(h))    # (1, 'list1')

True
True
False
(1, 'list1')


### 7.3 Cái bẫy: `ListNode` không so sánh được

Ta muốn đưa node vào heap theo `val`. Cách tự nhiên là tuple `(node.val, node)`. Nhưng khi **hai node có cùng `val`**, Python phải so tiếp phần tử thứ hai: `node_a < node_b` — mà class `ListNode` **không định nghĩa phép `<`** → `TypeError`.

Example 1 có hai số `1` và hai số `4`, nên lỗi này chắc chắn xảy ra!

In [14]:
a = ListNode(1)
b = ListNode(1)

h = []
heapq.heappush(h, (a.val, a))
try:
    heapq.heappush(h, (b.val, b))   # val bằng nhau -> phải so a với b
except TypeError as e:
    print("TypeError:", e)

TypeError: '<' not supported between instances of 'ListNode' and 'ListNode'


**Cách sửa**: chèn một số "phân xử" (tie-breaker) **duy nhất** vào giữa: `(val, idx, node)`.

- `idx` = chỉ số của list mà node đó thuộc về.
- Tại mọi thời điểm, mỗi list chỉ có **tối đa 1 node** trong heap → không có hai tuple nào trùng cả `val` lẫn `idx`.
- Vì vậy Python luôn phân thắng thua ở `val` hoặc `idx`, **không bao giờ** phải so tới `node`.

(Một cách khác: dùng bộ đếm tăng dần `itertools.count()` làm tie-breaker — cũng đảm bảo duy nhất.)

In [15]:
h = []
heapq.heappush(h, (a.val, 0, a))   # node a thuộc list 0
heapq.heappush(h, (b.val, 1, b))   # node b thuộc list 1 -> so 1 == 1, rồi 0 < 1, dừng
val, idx, node = heapq.heappop(h)
print("Lấy ra: val =", val, "từ list", idx, "| đúng là node a?", node is a)

Lấy ra: val = 1 từ list 0 | đúng là node a? True


## 8. Cách 4 — Min-heap (cách chính)

Ghép cách 3 với heap:

1. Đưa **node đầu** của mỗi list (nếu khác `None`) vào heap dưới dạng `(node.val, i, node)`.
2. Lặp khi heap còn phần tử:
   - `heappop` → node nhỏ nhất trong các "đầu list" hiện tại.
   - Gắn node đó vào đuôi kết quả (dummy head + tail).
   - Nếu `node.next` khác `None`, `heappush` node kế tiếp đó (cùng chỉ số `i`).
3. Trả `dummy.next`.

Heap luôn chứa **tối đa k phần tử** — đúng bằng "k đầu list" của cách 3, nhưng lấy min chỉ tốn `O(log k)` thay vì `O(k)`.

In [16]:
def merge_k_heap(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:                                   # bỏ qua list rỗng (None)
            heapq.heappush(heap, (node.val, i, node))

    dummy = ListNode()
    tail = dummy
    while heap:
        val, i, node = heapq.heappop(heap)         # node nhỏ nhất hiện tại
        tail.next = node
        tail = tail.next
        if node.next:                              # list i còn node -> đưa node kế vào heap
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


print(list_str(merge_k_heap(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))
print(list_str(merge_k_heap(build_lists([]))), list_str(merge_k_heap(build_lists([[]]))))

1->1->2->3->4->4->5->6
∅ ∅


### Trace cách 4 với Example 1

Heap ghi dưới dạng các cặp `(val, idx)` (bỏ phần `node` cho gọn), **xếp tăng dần để dễ đọc** (list thật bên trong heap có thể theo thứ tự khác).

Ban đầu heap = `[(1,0), (1,1), (2,2)]`.

| Bước | Pop | Push node kế? | Heap sau bước | Kết quả |
|---:|---|---|---|---|
| 1 | `(1,0)` | `(4,0)` | `[(1,1), (2,2), (4,0)]` | `1` |
| 2 | `(1,1)` | `(3,1)` | `[(2,2), (3,1), (4,0)]` | `1->1` |
| 3 | `(2,2)` | `(6,2)` | `[(3,1), (4,0), (6,2)]` | `1->1->2` |
| 4 | `(3,1)` | `(4,1)` | `[(4,0), (4,1), (6,2)]` | `...->3` |
| 5 | `(4,0)` | `(5,0)` | `[(4,1), (5,0), (6,2)]` | `...->4` |
| 6 | `(4,1)` | list 1 hết | `[(5,0), (6,2)]` | `...->4->4` |
| 7 | `(5,0)` | list 0 hết | `[(6,2)]` | `...->5` |
| 8 | `(6,2)` | list 2 hết | `[]` | `1->1->2->3->4->4->5->6` |

Bước 4–5 cho thấy vai trò của `idx`: hai tuple `(4,0)` và `(4,1)` bằng nhau ở `val`, heap phân xử bằng `idx` (0 < 1) mà không đụng tới `node`.

In [17]:
def trace_heap(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    show = lambda: sorted((v, i) for v, i, _ in heap)
    print("Ban đầu heap =", show())
    out = []
    step = 1
    while heap:
        val, i, node = heapq.heappop(heap)
        out.append(val)
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
            pushed = f"push ({node.next.val},{i})"
        else:
            pushed = f"list {i} hết"
        print(f"Bước {step}: pop ({val},{i}) | {pushed:12} | heap = {show()} | kết quả = {out}")
        step += 1


trace_heap(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))

Ban đầu heap = [(1, 0), (1, 1), (2, 2)]
Bước 1: pop (1,0) | push (4,0)   | heap = [(1, 1), (2, 2), (4, 0)] | kết quả = [1]
Bước 2: pop (1,1) | push (3,1)   | heap = [(2, 2), (3, 1), (4, 0)] | kết quả = [1, 1]
Bước 3: pop (2,2) | push (6,2)   | heap = [(3, 1), (4, 0), (6, 2)] | kết quả = [1, 1, 2]
Bước 4: pop (3,1) | push (4,1)   | heap = [(4, 0), (4, 1), (6, 2)] | kết quả = [1, 1, 2, 3]
Bước 5: pop (4,0) | push (5,0)   | heap = [(4, 1), (5, 0), (6, 2)] | kết quả = [1, 1, 2, 3, 4]
Bước 6: pop (4,1) | list 1 hết   | heap = [(5, 0), (6, 2)] | kết quả = [1, 1, 2, 3, 4, 4]
Bước 7: pop (5,0) | list 0 hết   | heap = [(6, 2)] | kết quả = [1, 1, 2, 3, 4, 4, 5]
Bước 8: pop (6,2) | list 2 hết   | heap = [] | kết quả = [1, 1, 2, 3, 4, 4, 5, 6]


### Vì sao cách 4 đúng? Độ phức tạp?

**Invariant** (điều luôn đúng trước mỗi lần pop): heap chứa đúng **node đầu của phần chưa lấy** của mỗi list còn node.

- Vì mỗi list tăng dần, node nhỏ nhất trong *toàn bộ* phần chưa lấy chắc chắn là một trong các node đầu đó → pop ra đúng node nhỏ nhất còn lại.
- Lấy node nhỏ nhất còn lại ở mỗi bước → kết quả tăng dần.
- Sau khi pop node của list `i`, ta push `node.next` → invariant được giữ. Không node nào bị bỏ sót (mọi node đều vào heap đúng một lần khi nó trở thành đầu list).

**Độ phức tạp** (`k` list, `N` node):

- Mỗi node được push 1 lần và pop 1 lần, heap có tối đa `k` phần tử → mỗi thao tác `O(log k)`.
- Time: `O(N log k)` (cộng `O(k log k)` lúc khởi tạo — có thể giảm còn `O(k)` bằng `heapify`).
- Space: `O(k)` cho heap; kết quả dùng lại node cũ.

## 9. Cách 5 — Chia để trị: ghép cặp (divide and conquer)

Quay lại cách 2: chậm vì `result` phình to rồi bị duyệt lại nhiều lần — kiểu gộp "lệch một bên".

Ý tưởng mới: gộp **theo cặp cho cân bằng**, giống giải đấu loại trực tiếp:

```text
Vòng 1:  [L0 L1] [L2 L3] [L4 L5] [L6 L7]   -> 4 list
Vòng 2:  [  A  B  ] [  C  D  ]              -> 2 list
Vòng 3:  [      E      F      ]             -> 1 list = đáp án
```

- Mỗi vòng, số list giảm một nửa → có khoảng `log2(k)` vòng.
- Trong một vòng, mỗi node tham gia **đúng một** lần merge → mỗi vòng tốn `O(N)`.

Nếu số list lẻ, list cuối "được miễn đấu", đi thẳng vào vòng sau.

In [18]:
def merge_k_divide_conquer(lists):
    if not lists:                        # k = 0
        return None
    current = list(lists)
    while len(current) > 1:
        merged = []
        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None   # list lẻ cuối ghép với None
            merged.append(merge_two_lists(a, b))
        current = merged
    return current[0]


print(list_str(merge_k_divide_conquer(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))
print(list_str(merge_k_divide_conquer(build_lists([]))), list_str(merge_k_divide_conquer(build_lists([[]]))))

1->1->2->3->4->4->5->6
∅ ∅


### Trace cách 5 với 5 list (số lẻ)

Input: `[[1,4,5], [1,3,4], [2,6], [0,7], [3]]`

| Vòng | Các list trước vòng | Các cặp gộp | Các list sau vòng |
|---:|---|---|---|
| 1 | `1->4->5`, `1->3->4`, `2->6`, `0->7`, `3` | (0,1), (2,3), (4, None) | `1->1->3->4->4->5`, `0->2->6->7`, `3` |
| 2 | 3 list ở trên | (0,1), (2, None) | `0->1->1->2->3->4->4->5->6->7`, `3` |
| 3 | 2 list ở trên | (0,1) | `0->1->1->2->3->3->4->4->5->6->7` |

In [19]:
def trace_divide_conquer(lists):
    current = list(lists)
    rnd = 1
    while len(current) > 1:
        print(f"Vòng {rnd}: trước = {[list_str(x) for x in current]}")
        merged = []
        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None
            merged.append(merge_two_lists(a, b))
        current = merged
        print(f"        sau   = {[list_str(x) for x in current]}")
        rnd += 1


trace_divide_conquer(build_lists([[1, 4, 5], [1, 3, 4], [2, 6], [0, 7], [3]]))

Vòng 1: trước = ['1->4->5', '1->3->4', '2->6', '0->7', '3']
        sau   = ['1->1->3->4->4->5', '0->2->6->7', '3']
Vòng 2: trước = ['1->1->3->4->4->5', '0->2->6->7', '3']
        sau   = ['0->1->1->2->3->4->4->5->6->7', '3']
Vòng 3: trước = ['0->1->1->2->3->4->4->5->6->7', '3']
        sau   = ['0->1->1->2->3->3->4->4->5->6->7']


### Vì sao cách 5 đúng? Độ phức tạp?

**Invariant**: sau mỗi vòng, mỗi phần tử của `current` là một linked list **đã sắp xếp**, và tập tất cả node trong `current` **đúng bằng** tập node ban đầu (không mất, không thêm).

- Vòng đầu: mỗi list đã sorted theo đề.
- `merge_two_lists` của hai list sorted cho ra list sorted chứa đủ node của cả hai → invariant giữ qua mỗi vòng.
- Khi chỉ còn 1 list → nó chứa mọi node và đã sorted.

**Độ phức tạp**:

- Số vòng: `⌈log2 k⌉`. Mỗi vòng, mỗi node đi qua tối đa một lần merge → `O(N)` mỗi vòng.
- Time: `O(N log k)`.
- Space: `O(k)` cho mảng con trỏ `current`/`merged` (không tạo node mới).

Cùng ý tưởng cũng viết được bằng **đệ quy** (recursion): "gộp `lists[lo..hi]` = gộp nửa trái + gộp nửa phải rồi merge two" — chính là khung **merge sort**. Bản vòng lặp ở trên dễ theo dõi hơn cho người mới nên ta dùng nó.

## 10. Code nộp LeetCode

### 10.1 Bản chính — Min-heap

Trên LeetCode **không cần** định nghĩa lại `ListNode` hay import `List`, `Optional` (đã có sẵn). Chỉ cần `import heapq` (thực ra LeetCode cũng import sẵn, nhưng ghi rõ cho chắc).

Bỏ hết `print`, giữ code gọn:

In [20]:
import heapq


class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        heap = []
        for i, node in enumerate(lists):
            if node:
                heapq.heappush(heap, (node.val, i, node))

        dummy = ListNode()
        tail = dummy
        while heap:
            _, i, node = heapq.heappop(heap)
            tail.next = node
            tail = tail.next
            if node.next:
                heapq.heappush(heap, (node.next.val, i, node.next))
        return dummy.next


sol = Solution()
assert to_list(sol.mergeKLists(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))) == [1, 1, 2, 3, 4, 4, 5, 6]
assert sol.mergeKLists(build_lists([])) is None
assert sol.mergeKLists(build_lists([[]])) is None
assert to_list(sol.mergeKLists(build_lists([[], [], [7]]))) == [7]               # nhiều list rỗng
assert to_list(sol.mergeKLists(build_lists([[2, 2], [2], [2, 2, 2]]))) == [2] * 6  # toàn số trùng
assert to_list(sol.mergeKLists(build_lists([[-10000, 10000], [-10000]]))) == [-10000, -10000, 10000]
assert to_list(sol.mergeKLists(build_lists([[5, 6, 7]]))) == [5, 6, 7]           # k = 1
print("Solution (heap): PASS các ví dụ đề + edge case")

Solution (heap): PASS các ví dụ đề + edge case


### 10.2 Bản phụ — Chia để trị ghép cặp

Cũng `O(N log k)`, không cần heap, và luyện lại merge two. Đặt tên class khác để chạy cùng notebook (khi nộp thì đổi lại thành `Solution`).

Trong `mergeTwo` có dòng `tail.next, a = a, a.next` — **gán đồng thời**: Python tính hết vế phải (`a` và `a.next` *cũ*) trước, rồi mới gán. Nó tương đương hai dòng `tail.next = a` rồi `a = a.next`.

In [21]:
class SolutionDivideConquer:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        if not lists:
            return None
        while len(lists) > 1:
            merged = []
            for i in range(0, len(lists), 2):
                b = lists[i + 1] if i + 1 < len(lists) else None
                merged.append(self.mergeTwo(lists[i], b))
            lists = merged
        return lists[0]

    def mergeTwo(self, a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode()
        tail = dummy
        while a and b:
            if a.val <= b.val:
                tail.next, a = a, a.next
            else:
                tail.next, b = b, b.next
            tail = tail.next
        tail.next = a if a else b
        return dummy.next


sol2 = SolutionDivideConquer()
assert to_list(sol2.mergeKLists(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))) == [1, 1, 2, 3, 4, 4, 5, 6]
assert sol2.mergeKLists(build_lists([])) is None
assert sol2.mergeKLists(build_lists([[]])) is None
assert to_list(sol2.mergeKLists(build_lists([[], [], [7]]))) == [7]
assert to_list(sol2.mergeKLists(build_lists([[2, 2], [2], [2, 2, 2]]))) == [2] * 6
print("SolutionDivideConquer: PASS")

SolutionDivideConquer: PASS


## 11. Test đối chiếu chéo trên input ngẫu nhiên

Ta chạy cả 5 cách + 2 class Solution trên hàng trăm input nhỏ ngẫu nhiên, so với đáp án "chắc chắn đúng": `sorted(mọi giá trị)`.

Hai điểm quan trọng:

- `random.seed(23)` cố định để lần chạy nào cũng ra cùng input (dễ tái hiện lỗi).
- Các cách 2–5 **sửa `next` của node đầu vào**, nên mỗi lần gọi phải `build_lists(raw)` **mới**. Nếu dùng chung, cách chạy sau sẽ nhận input đã bị "xáo dây".

In [22]:
import random

random.seed(23)

methods = {
    "1_sort_values": merge_k_sort_values,
    "2_one_by_one": merge_k_one_by_one,
    "3_compare_heads": merge_k_compare_heads,
    "4_heap": merge_k_heap,
    "5_divide_conquer": merge_k_divide_conquer,
    "Solution": Solution().mergeKLists,
    "SolutionDivideConquer": SolutionDivideConquer().mergeKLists,
}

num_tests = 500
for _ in range(num_tests):
    k = random.randint(0, 7)
    raw = [sorted(random.randint(-5, 5) for _ in range(random.randint(0, 6))) for _ in range(k)]
    expected = sorted(v for arr in raw for v in arr)
    for name, f in methods.items():
        got = to_list(f(build_lists(raw)))      # input mới cho mỗi cách
        assert got == expected, (name, raw, got, expected)

print(f"{len(methods)} cách x {num_tests} test ngẫu nhiên: tất cả khớp nhau và khớp sorted()")

# Test cỡ lớn theo constraints: k = 10^4 list, mỗi list 1 node (N = 10^4) và k = 20 list x 500 node
random.seed(7)
big1 = [[random.randint(-10**4, 10**4)] for _ in range(10**4)]
big2 = [sorted(random.randint(-10**4, 10**4) for _ in range(500)) for _ in range(20)]

for raw in (big1, big2):
    expected = sorted(v for arr in raw for v in arr)
    assert to_list(Solution().mergeKLists(build_lists(raw))) == expected
    assert to_list(SolutionDivideConquer().mergeKLists(build_lists(raw))) == expected
    print(f"k = {len(raw):>5}, N = {len(expected)}: PASS")

7 cách x 500 test ngẫu nhiên: tất cả khớp nhau và khớp sorted()
k = 10000, N = 10000: PASS
k =    20, N = 10000: PASS


## 12. Bảng so sánh các cách

`k` = số list, `N` = tổng số node.

| Cách | Ý tưởng | Time | Space phụ | Nhận xét |
|---|---|---|---|---|
| 1. Gom + sort | lấy hết val, `sort`, dựng lại | `O(N log N)` | `O(N)` | dễ nhất, bỏ phí "đã sorted" |
| 2. Merge lần lượt | `result = merge(result, list_i)` | `O(k·N)` | `O(1)` | kết quả phình to bị duyệt lại → dễ TLE |
| 3. So `k` đầu | quét `k` đầu list tìm min | `O(k·N)` | `O(k)` | tìm min `O(k)` mỗi bước |
| 4. Min-heap | heap giữ `k` đầu list | `O(N log k)` | `O(k)` | **cách chính**, dễ mở rộng |
| 5. Chia để trị | ghép cặp từng vòng | `O(N log k)` | `O(k)` | không cần heap, khung merge sort |

Để cảm nhận sự khác biệt, ta **ước lượng số bước** (số node phải đi qua / số phép so sánh) theo đúng công thức của từng cách, cho vài cấu hình `k` list, mỗi list `n` node. Đây là đếm theo công thức, không đo thời gian thật nên kết quả luôn cố định.

Hãy chú ý dòng `k = 10^4` (mỗi list 1 node): cách 2 và 3 cần cỡ `5·10^7`–`10^8` bước, còn heap và chia để trị chỉ cỡ `1.4·10^5`. Đó là khác biệt giữa `O(k·N)` và `O(N log k)`.

In [23]:
import math


def estimate_steps(k, n):
    N = k * n
    one_by_one = sum(i * n + n for i in range(k))     # lần i: result dài i*n, thêm n node
    compare_heads = N * k                             # N bước, mỗi bước quét k đầu
    log_k = max(1, math.ceil(math.log2(k))) if k > 1 else 1
    heap = N * log_k                                  # mỗi node push + pop ~ log k
    dc = N * log_k                                    # log k vòng, mỗi vòng ~ N
    return N, one_by_one, compare_heads, heap, dc


print(f"{'k':>6} {'n':>4} {'N':>6} | {'2.lần lượt':>11} {'3.so k đầu':>11} {'4.heap':>9} {'5.chia trị':>10}")
for k, n in [(4, 25), (100, 100), (10**4, 1)]:
    N, a, b, c, d = estimate_steps(k, n)
    print(f"{k:>6} {n:>4} {N:>6} | {a:>11,} {b:>11,} {c:>9,} {d:>10,}")

     k    n      N |  2.lần lượt  3.so k đầu    4.heap 5.chia trị
     4   25    100 |         250         400       200        200
   100  100  10000 |     505,000   1,000,000    70,000     70,000
 10000    1  10000 |  50,005,000 100,000,000   140,000    140,000


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đẩy `(val, node)` vào heap

```python
heapq.heappush(heap, (node.val, node))   # SAI khi có val trùng
```

Khi hai `val` bằng nhau, Python so `node < node` → `TypeError: '<' not supported between instances of 'ListNode' and 'ListNode'` (đã thấy ở mục 7.3). **Sửa**: `(node.val, i, node)` với `i` duy nhất.

### Lỗi 2: Trả `dummy` thay vì `dummy.next`

Kết quả bị thừa một node giả ở đầu.

In [24]:
def merge_k_heap_wrong_return(lists):
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = ListNode()          # val mặc định = 0
    tail = dummy
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy                # SAI: phải là dummy.next


print("Sai :", to_list(merge_k_heap_wrong_return(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))
print("Đúng:", to_list(merge_k_heap(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))))

Sai : [0, 1, 1, 2, 3, 4, 4, 5, 6]
Đúng: [1, 1, 2, 3, 4, 4, 5, 6]


(Ở cell trên ta dựng heap bằng `heapify` một lần cho cả `k` phần tử — `O(k)` — cách khởi tạo này hoàn toàn hợp lệ; lỗi chỉ nằm ở `return`.)

### Lỗi 3: Quên đẩy `node.next` vào heap

Nếu sau khi pop quên `heappush(node.next)`, heap chỉ chứa `k` node đầu → vòng lặp dừng sau tối đa `k` lần pop, kết quả sai.

### Lỗi 4: Quên `tail = tail.next`

Nếu chỉ viết `tail.next = node` mà không dịch `tail`, `tail` mãi đứng ở dummy, nên `dummy.next` bị ghi đè liên tục bằng node mới nhất → kết quả sai (chỉ còn node lấy sau cùng cùng phần đuôi cũ của nó).

### Lỗi 5: Cách 2 với `k` lớn bị TLE

Code cách 2 đúng logic, nhưng `O(k·N)` có thể quá chậm khi `k = 10^4`. Nếu đã biết merge two, hãy chuyển sang **ghép cặp** (cách 5) — chỉ đổi thứ tự merge mà nhanh hơn hẳn.

### Lỗi 6: Quên bỏ qua list rỗng

Với input `[[]]`, `lists = [None]`. Nếu viết `heappush(heap, (node.val, i, node))` mà không có `if node:` → `AttributeError: 'NoneType' object has no attribute 'val'`.

In [25]:
def init_heap_wrong(lists):
    heap = []
    for i, node in enumerate(lists):
        heapq.heappush(heap, (node.val, i, node))   # SAI: thiếu if node
    return heap


try:
    init_heap_wrong(build_lists([[]]))
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'val'


### Lỗi 7: Dùng lại input sau khi đã merge

Các cách 2–5 nối lại `next` của node gốc. List đầu vào sau khi merge **không còn như cũ**, nên khi test nhiều cách phải `build_lists(raw)` mới cho mỗi lần gọi (như mục 11):

In [26]:
x = build_list([1, 4, 5])
y = build_list([1, 3, 4])
print("x trước merge:", list_str(x))
merge_two_lists(x, y)
print("x sau merge  :", list_str(x))   # x vẫn là node 1 đầu tiên, nhưng chuỗi next đã bị nối lại

x trước merge: 1->4->5
x sau merge  : 1->1->3->4->4->5


## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu "k-way merge" (gộp k nguồn)

- Có **nhiều nguồn dữ liệu, mỗi nguồn đã sắp xếp**, cần sinh ra kết quả theo thứ tự chung.
- Cần "phần tử nhỏ nhất kế tiếp" liên tục trong một tập **thay đổi từng chút** → nghĩ ngay tới **heap**.
- Thấy cách "gộp dần vào một kết quả" bị lệch → nghĩ tới **ghép cặp cân bằng** (chia để trị, khung merge sort).

### Template trong đầu

```text
heap = [(đầu nguồn i, i, con trỏ)] cho mọi nguồn khác rỗng
while heap:
    pop phần tử nhỏ nhất -> dùng nó
    nếu nguồn đó còn phần tử kế -> push phần tử kế
```

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 21. Merge Two Sorted Lists | chính là viên gạch `merge_two_lists` |
| 88. Merge Sorted Array | merge two trên mảng |
| 148. Sort List | merge sort trên linked list (chia để trị) |
| 378. Kth Smallest Element in a Sorted Matrix | mỗi hàng là một nguồn sorted → k-way merge bằng heap |
| 373. Find K Pairs with Smallest Sums | heap trên các "nguồn" cặp số |
| 632. Smallest Range Covering Elements from K Lists | heap giữ k đầu list + theo dõi max |
| 264. Ugly Number II | gộp nhiều dãy tăng dần (nhân 2, 3, 5) |

Ngoài LeetCode: **external sort** (sắp xếp file lớn hơn RAM) chia file thành nhiều đoạn đã sort rồi gộp k đoạn bằng heap — đúng bài này. Thư viện chuẩn Python còn có sẵn `heapq.merge(*iterables)` làm k-way merge cho các iterable đã sắp xếp: `list(heapq.merge([1,4,5], [1,3,4], [2,6]))` cho `[1, 1, 2, 3, 4, 4, 5, 6]` (dùng cho Python list; với `ListNode` vẫn phải tự viết như bài này).

## 15. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý ở mục 16, đáp án ở mục 17 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

```python
import heapq
h = []
for x in [5, 1, 4, 1]:
    heapq.heappush(h, x)
print(h[0])
print(heapq.heappop(h), heapq.heappop(h))
print(len(h))
print((3, 0) < (3, 1), (2, 9) < (3, 0))
```

Viết ra 4 dòng output và giải thích từng dòng.

### Bài 2 — Tính tay bằng bảng (cách 4)

Cho `lists = [[2,5], [1,9], [3]]`. Lập bảng giống mục 8: mỗi bước ghi tuple `(val, idx)` bị pop, tuple được push (nếu có), heap sau bước (xếp tăng dần), và kết quả.

Có bao nhiêu bước? Kết quả cuối là gì?

### Bài 3 — Điền TODO: merge two sorted lists

Hoàn thiện hàm gộp hai linked list đã sắp xếp, dùng dummy head, không tạo node mới.

In [27]:
def merge_two_exercise(a, b):
    dummy = ListNode()
    tail = dummy
    # TODO 1: while cả a và b còn node
    # TODO 2:     gắn node nhỏ hơn vào tail.next, con trỏ đó tiến lên
    # TODO 3:     dịch tail
    # TODO 4: gắn phần còn lại (a hoặc b)
    # TODO 5: trả về node thật đầu tiên
    pass


# Tự làm xong rồi mới mở test:
# assert to_list(merge_two_exercise(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
# assert to_list(merge_two_exercise(None, build_list([0]))) == [0]
# assert merge_two_exercise(None, None) is None

### Bài 4 — Tự viết lại: Merge k lists bằng min-heap

Không nhìn lại mục 8, tự viết hàm với đúng signature của LeetCode (dạng hàm). Nhớ: tuple 3 phần tử, bỏ qua list rỗng, push node kế, trả `dummy.next`.

In [28]:
def merge_k_heap_exercise(lists):
    # TODO: viết lại cách min-heap
    raise NotImplementedError


# Gợi ý test:
# assert to_list(merge_k_heap_exercise(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]))) == [1, 1, 2, 3, 4, 4, 5, 6]
# assert merge_k_heap_exercise(build_lists([])) is None
# assert merge_k_heap_exercise(build_lists([[]])) is None
# assert to_list(merge_k_heap_exercise(build_lists([[2, 2], [2]]))) == [2, 2, 2]

### Bài 5 — Biến thể: gộp k **mảng** Python đã sắp xếp

Lần này input là list các Python list (không phải linked list), trả về một Python list đã sắp xếp. Không được gom hết rồi `sort` — hãy dùng heap.

Gợi ý nhỏ: thay vì con trỏ `node`, mỗi nguồn cần một **chỉ số vị trí** `j` trong mảng `arrays[i]`.

```python
merge_k_sorted_arrays([[1,4,5],[1,3,4],[2,6]]) -> [1,1,2,3,4,4,5,6]
merge_k_sorted_arrays([]) -> []
merge_k_sorted_arrays([[], [0]]) -> [0]
```

In [29]:
def merge_k_sorted_arrays(arrays):
    # TODO: heap chứa tuple (giá trị, i, j) với arrays[i][j] là giá trị đó
    pass


# Gợi ý test:
# assert merge_k_sorted_arrays([[1, 4, 5], [1, 3, 4], [2, 6]]) == [1, 1, 2, 3, 4, 4, 5, 6]
# assert merge_k_sorted_arrays([]) == []
# assert merge_k_sorted_arrays([[], [0]]) == [0]

### Bài 6 — Mini challenge: phần tử nhỏ thứ `t` trong k linked list

Cho `k` linked list đã sắp xếp và số nguyên `t` (`1 <= t <= N`). Trả về **giá trị** nhỏ thứ `t` (đếm cả giá trị trùng), **không** dựng toàn bộ list gộp.

Yêu cầu độ phức tạp: `O(k log k + t·log k)` (hoặc `O(k + t·log k)` nếu khởi tạo heap bằng `heapify`), tức là **không** phải duyệt hết `N` node khi `t` nhỏ.

```python
lists = [[1,4,5],[1,3,4],[2,6]]
kth_smallest_in_lists(lists, 1) -> 1
kth_smallest_in_lists(lists, 3) -> 2
kth_smallest_in_lists(lists, 8) -> 6
```

In [30]:
def kth_smallest_in_lists(lists, t):
    # TODO: pop từ heap t lần, lần thứ t chính là đáp án
    pass


# Gợi ý test:
# assert kth_smallest_in_lists(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]), 1) == 1
# assert kth_smallest_in_lists(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]), 3) == 2
# assert kth_smallest_in_lists(build_lists([[1, 4, 5], [1, 3, 4], [2, 6]]), 8) == 6

## 16. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Heap luôn giữ nhỏ nhất ở `h[0]`. | `heappop` vừa trả vừa xóa phần tử nhỏ nhất. | Có hai số `1`; sau 2 lần pop còn lại `{4, 5}`. |
| 2 | Heap ban đầu chứa đầu của 3 list. | Ban đầu: `(2,0), (1,1), (3,2)`. | Pop `(1,1)` trước, push `(9,1)`. |
| 3 | Giống hệt hai con trỏ ở mục 3. | `while a and b:`, `if a.val <= b.val:` | `tail.next = a if a else b`; `return dummy.next` |
| 4 | Heap giữ "k đầu list". | Tuple `(node.val, i, node)`; `if node:` khi khởi tạo. | Sau `heappop`: `tail.next = node; tail = tail.next; if node.next: heappush(...)` |
| 5 | "Con trỏ" bây giờ là chỉ số `j`. | Push `(arrays[i][0], i, 0)` cho mảng không rỗng. | Pop `(v, i, j)` → `out.append(v)`; nếu `j + 1 < len(arrays[i])` thì push `(arrays[i][j+1], i, j+1)`. |
| 6 | Không cần dựng list: chỉ cần đếm số lần pop. | Vòng `for _ in range(t)`. | Lần pop thứ `t` → trả `val` của lần đó. |

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
1
1 1
2
True True
```

- Push `5, 1, 4, 1` → nhỏ nhất là `1`, nên `h[0] = 1`.
- Có **hai** số `1`, pop hai lần ra `1 1`.
- Ban đầu 4 phần tử, pop 2 → còn `2`.
- `(3, 0) < (3, 1)`: phần đầu bằng nhau, `0 < 1` → `True`. `(2, 9) < (3, 0)`: `2 < 3` → `True` ngay, không cần xem `9`.

### Đáp án Bài 2

Ban đầu heap = `[(1,1), (2,0), (3,2)]`.

| Bước | Pop | Push | Heap sau bước | Kết quả |
|---:|---|---|---|---|
| 1 | `(1,1)` | `(9,1)` | `[(2,0), (3,2), (9,1)]` | `[1]` |
| 2 | `(2,0)` | `(5,0)` | `[(3,2), (5,0), (9,1)]` | `[1, 2]` |
| 3 | `(3,2)` | list 2 hết | `[(5,0), (9,1)]` | `[1, 2, 3]` |
| 4 | `(5,0)` | list 0 hết | `[(9,1)]` | `[1, 2, 3, 5]` |
| 5 | `(9,1)` | list 1 hết | `[]` | `[1, 2, 3, 5, 9]` |

Có 5 bước (đúng bằng `N = 5` node). Cell kế tiếp kiểm chứng Bài 2 bằng `trace_heap`; cell sau nữa là **đáp án Bài 3–6** kèm assert.

Nhận xét Bài 6: chỉ pop `t` lần nên tốn `O(k log k + t·log k)` (khởi tạo bằng `heapify` thì phần đầu còn `O(k)`). Khi `t` nhỏ hơn nhiều so với `N`, cách này nhanh hơn hẳn việc merge hết rồi lấy phần tử thứ `t`.

In [31]:
trace_heap(build_lists([[2, 5], [1, 9], [3]]))

Ban đầu heap = [(1, 1), (2, 0), (3, 2)]
Bước 1: pop (1,1) | push (9,1)   | heap = [(2, 0), (3, 2), (9, 1)] | kết quả = [1]
Bước 2: pop (2,0) | push (5,0)   | heap = [(3, 2), (5, 0), (9, 1)] | kết quả = [1, 2]
Bước 3: pop (3,2) | list 2 hết   | heap = [(5, 0), (9, 1)] | kết quả = [1, 2, 3]
Bước 4: pop (5,0) | list 0 hết   | heap = [(9, 1)] | kết quả = [1, 2, 3, 5]
Bước 5: pop (9,1) | list 1 hết   | heap = [] | kết quả = [1, 2, 3, 5, 9]


In [32]:
def answer_merge_two_exercise(a, b):
    dummy = ListNode()
    tail = dummy
    while a and b:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next


def answer_merge_k_heap_exercise(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = ListNode()
    tail = dummy
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


def answer_merge_k_sorted_arrays(arrays):
    heap = []
    for i, arr in enumerate(arrays):
        if arr:
            heapq.heappush(heap, (arr[0], i, 0))
    out = []
    while heap:
        v, i, j = heapq.heappop(heap)
        out.append(v)
        if j + 1 < len(arrays[i]):
            heapq.heappush(heap, (arrays[i][j + 1], i, j + 1))
    return out


def answer_kth_smallest_in_lists(lists, t):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    for _ in range(t):
        val, i, node = heapq.heappop(heap)
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return val


ex1 = [[1, 4, 5], [1, 3, 4], [2, 6]]

assert to_list(answer_merge_two_exercise(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
assert to_list(answer_merge_two_exercise(None, build_list([0]))) == [0]
assert answer_merge_two_exercise(None, None) is None

assert to_list(answer_merge_k_heap_exercise(build_lists(ex1))) == [1, 1, 2, 3, 4, 4, 5, 6]
assert answer_merge_k_heap_exercise(build_lists([])) is None
assert answer_merge_k_heap_exercise(build_lists([[]])) is None
assert to_list(answer_merge_k_heap_exercise(build_lists([[2, 2], [2]]))) == [2, 2, 2]

assert answer_merge_k_sorted_arrays(ex1) == [1, 1, 2, 3, 4, 4, 5, 6]
assert answer_merge_k_sorted_arrays([]) == []
assert answer_merge_k_sorted_arrays([[], [0]]) == [0]

assert answer_kth_smallest_in_lists(build_lists(ex1), 1) == 1
assert answer_kth_smallest_in_lists(build_lists(ex1), 3) == 2
assert answer_kth_smallest_in_lists(build_lists(ex1), 8) == 6

# Đối chiếu thêm với input ngẫu nhiên
random.seed(2023)
for _ in range(200):
    raw = [sorted(random.randint(-9, 9) for _ in range(random.randint(0, 5))) for _ in range(random.randint(0, 5))]
    flat = sorted(v for arr in raw for v in arr)
    assert answer_merge_k_sorted_arrays(raw) == flat
    assert to_list(answer_merge_k_heap_exercise(build_lists(raw))) == flat
    for t in range(1, len(flat) + 1):
        assert answer_kth_smallest_in_lists(build_lists(raw), t) == flat[t - 1]

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 18. Template cuối cùng nên nhớ

**Min-heap (cách chính)** — `O(N log k)` time, `O(k)` space:

```python
import heapq

class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        heap = []
        for i, node in enumerate(lists):
            if node:
                heapq.heappush(heap, (node.val, i, node))
        dummy = tail = ListNode()
        while heap:
            _, i, node = heapq.heappop(heap)
            tail.next = node
            tail = tail.next
            if node.next:
                heapq.heappush(heap, (node.next.val, i, node.next))
        return dummy.next
```

**Câu thần chú**: *"Heap giữ k cái đầu — pop nhỏ nhất, nối vào đuôi, push đứa kế tiếp."*

**Chia để trị**: *"Ghép cặp từng vòng bằng merge two, số list giảm một nửa mỗi vòng."*

(`dummy = tail = ListNode()` gán cùng một node cho cả hai tên — viết gọn của hai dòng `dummy = ListNode()` và `tail = dummy`.)

## 19. Checklist tự kiểm tra trước khi submit

- Mình có xử lý `lists = []` (k = 0) → trả `None` chưa?
- Mình có bỏ qua phần tử `None` (list rỗng, ví dụ `[[]]`) trước khi đọc `.val` chưa?
- Tuple trong heap có tie-breaker duy nhất `(val, i, node)` để không bao giờ so sánh hai `ListNode` chưa?
- Sau mỗi lần pop, mình có push `node.next` (khi khác `None`) chưa?
- Có `tail = tail.next` sau mỗi lần nối node chưa?
- Mình trả `dummy.next` (không phải `dummy`) chưa?
- Đã test giá trị trùng nhau, list rỗng xen giữa, `k = 1`, giá trị âm chưa?
- Nếu dùng merge lần lượt, mình có ý thức là `O(k·N)` dễ TLE và nên đổi sang ghép cặp / heap chưa?